# Online Shoppers Purchasing Intention — Phase 1
## EDA · Preprocessing · Feature Engineering

Dataset: 12,330 online shopping sessions · 18 features · Target: `Revenue` (True/False)

This notebook merges the three Phase 1 sections into one pipeline:
1. **Exploratory Data Analysis** — understanding the raw data
2. **Preprocessing** — cleaning, transforming, and encoding
3. **Feature Engineering** — new features, binning, selection, correlation analysis
4. **Final split, scaling, and export** — ready for Phase 2 modeling

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from matplotlib.lines import Line2D
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

COLORS = {'no': '#e74c3c', 'yes': '#2ecc71'}
PALETTE = [COLORS['no'], COLORS['yes']]

df = pd.read_csv('online_shoppers_intention.csv')

# Convert string booleans to actual booleans
df['Revenue'] = df['Revenue'].astype(str).str.strip().map(
    {'True': True, 'TRUE': True, 'False': False, 'FALSE': False})
df['Weekend'] = df['Weekend'].astype(str).str.strip().map(
    {'True': True, 'TRUE': True, 'False': False, 'FALSE': False})

print(f'Dataset shape: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head()

# Part 1 — Exploratory Data Analysis

## 1.1 Basic Info — Shape, Datatypes, Statistical Summary

In [ ]:
print("SHAPE:", df.shape)
print("\nCOLUMN NAMES:")
print(df.columns.tolist())
print("\nDATATYPES:")
print(df.dtypes)
print("\nDATASET INFO:")
df.info()

In [ ]:
print("STATISTICAL SUMMARY (Numeric):")
df.describe().round(2)

In [ ]:
# Categorical summaries
for col in ['Month', 'VisitorType']:
    print(f'\n{col} ({df[col].nunique()} unique):')
    print(df[col].value_counts())

for col in ['Weekend', 'Revenue']:
    print(f'\n{col}:')
    print(df[col].value_counts())

## 1.2 Missing Values & Duplicates

In [ ]:
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
missing = missing[missing['Missing Count'] > 0]

if missing.empty:
    print("No missing values found in any column.")
else:
    print("Missing Values:")
    print(missing.sort_values('Missing %', ascending=False))

plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=True, yticklabels=False, cmap='viridis')
plt.title("Missing Values Heatmap")
plt.tight_layout()
plt.show()

dup_count = df.duplicated().sum()
print(f"\nDuplicate Rows: {dup_count} ({dup_count/len(df)*100:.2f}%)")

## 1.3 Zero-Value Analysis

In [ ]:
print('Percentage of zero values per numerical column:')
print('=' * 45)
num_cols = df.select_dtypes(include=[np.number]).columns
for col in num_cols:
    zero_pct = (df[col] == 0).mean() * 100
    if zero_pct > 10:
        print(f'  {col:<30} {zero_pct:>6.1f}%')

## 1.4 Target Balance (Revenue)

In [ ]:
print("Target Variable Distribution:")
print(df['Revenue'].value_counts())
counts = df['Revenue'].value_counts()
print(f'\nPurchase rate: {df["Revenue"].mean()*100:.1f}%')
print(f'Imbalance ratio: {counts[False]/counts[True]:.1f} : 1')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.countplot(x='Revenue', data=df, ax=axes[0], hue='Revenue',
              palette={False: '#60a5fa', True: '#34d399'}, legend=False)
axes[0].set_title("Revenue Distribution (Count)")
for p in axes[0].patches:
    axes[0].annotate(f'{int(p.get_height())}',
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontweight='bold')

df['Revenue'].value_counts().plot.pie(
    autopct='%1.1f%%', ax=axes[1],
    colors=['#60a5fa', '#34d399'],
    labels=['No Purchase', 'Purchase'],
    startangle=90, explode=[0, 0.05]
)
axes[1].set_title("Revenue Distribution (%)")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()

## 1.5 Univariate & Bivariate Distributions

In [ ]:
numeric_cols = df.select_dtypes(include='number').columns.tolist()

fig, axes = plt.subplots(4, 4, figsize=(18, 14))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    df[col].hist(bins=30, ax=axes[i], color='#60a5fa', edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].set_ylabel("Frequency")

for j in range(len(numeric_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle("Histograms of Numeric Features", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
cat_cols = ['Month', 'VisitorType']

fig, axes = plt.subplots(1, len(cat_cols), figsize=(14, 5))
for i, col in enumerate(cat_cols):
    df[col].value_counts().plot.bar(ax=axes[i], color='#f472b6', edgecolor='black', alpha=0.7)
    axes[i].set_title(col, fontsize=11, fontweight='bold')
    axes[i].tick_params(axis='x', rotation=45)

plt.suptitle("Bar Charts of Categorical Features", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Page and duration distributions split by Revenue
page_cols = ['Administrative', 'Informational', 'ProductRelated']
duration_cols = ['Administrative_Duration', 'Informational_Duration', 'ProductRelated_Duration']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
for i, col in enumerate(page_cols + duration_cols):
    row, c = i // 3, i % 3
    sns.histplot(data=df, x=col, hue='Revenue', bins=50, ax=axes[row][c],
                 palette=PALETTE, alpha=0.6, kde=False)
    axes[row][c].set_title(f'{col}', fontsize=11)
    q99 = df[col].quantile(0.99)
    axes[row][c].set_xlim(right=q99)

plt.suptitle('Distribution of Page & Duration Features by Purchase', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Google Analytics metrics split by Revenue
ga_cols = ['BounceRates', 'ExitRates', 'PageValues']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(ga_cols):
    sns.histplot(data=df, x=col, hue='Revenue', bins=50, ax=axes[i],
                 palette=PALETTE, alpha=0.6, kde=True)
    axes[i].set_title(f'{col} by Purchase', fontsize=12)

plt.tight_layout()
plt.show()

print('PageValues statistics by Revenue:')
print(df.groupby('Revenue')['PageValues'].describe().round(2))

## 1.6 Skewness Detection

In [ ]:
skewness = df[numeric_cols].skew().sort_values(ascending=False)

print("SKEWNESS VALUES:")
print("-" * 45)
for col, val in skewness.items():
    if abs(val) < 0.5:
        label = "Approx. Symmetric"
    elif abs(val) < 1:
        label = "Moderately Skewed"
    else:
        label = "Highly Skewed"
    print(f"  {col:<30} {val:>8.4f}  ({label})")

fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#f87171' if abs(v) > 1 else '#fbbf24' if abs(v) > 0.5 else '#34d399'
          for v in skewness.values]
skewness.plot.barh(ax=ax, color=colors, edgecolor='black')
ax.axvline(x=0, color='black', linewidth=1, linestyle='--')
ax.axvline(x=0.5, color='#fbbf24', linewidth=1, linestyle='--', alpha=0.5)
ax.axvline(x=-0.5, color='#fbbf24', linewidth=1, linestyle='--', alpha=0.5)
ax.set_title("Skewness of Numeric Features", fontsize=14, fontweight='bold')
ax.set_xlabel("Skewness")
plt.tight_layout()
plt.show()

## 1.7 Boxplots — Features by Revenue

In [ ]:
key_features = ['BounceRates', 'ExitRates', 'PageValues', 'ProductRelated',
                'Administrative', 'ProductRelated_Duration',
                'Administrative_Duration', 'Informational_Duration']

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
axes = axes.flatten()

for i, col in enumerate(key_features):
    sns.boxplot(x='Revenue', y=col, data=df, ax=axes[i],
                hue='Revenue', palette={False: '#60a5fa', True: '#34d399'}, legend=False)
    axes[i].set_title(f'{col} by Revenue', fontsize=11, fontweight='bold')

plt.suptitle("Boxplots: Features by Revenue", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 1.8 Scatter Plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

scatter_colors = df['Revenue'].map({True: '#34d399', False: '#60a5fa'})

axes[0].scatter(df['BounceRates'], df['ExitRates'], c=scatter_colors, alpha=0.3, s=10)
axes[0].set_xlabel('BounceRates')
axes[0].set_ylabel('ExitRates')
axes[0].set_title('BounceRates vs ExitRates', fontweight='bold')

axes[1].scatter(df['ProductRelated'], df['ProductRelated_Duration'],
                c=scatter_colors, alpha=0.3, s=10)
axes[1].set_xlabel('ProductRelated')
axes[1].set_ylabel('ProductRelated_Duration')
axes[1].set_title('ProductRelated vs Duration', fontweight='bold')

axes[2].scatter(df['PageValues'], df['ExitRates'], c=scatter_colors, alpha=0.3, s=10)
axes[2].set_xlabel('PageValues')
axes[2].set_ylabel('ExitRates')
axes[2].set_title('PageValues vs ExitRates', fontweight='bold')

legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#60a5fa',
           markersize=8, label='No Purchase'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#34d399',
           markersize=8, label='Purchase')
]
axes[2].legend(handles=legend_elements, loc='upper right')

plt.suptitle("Scatter Plots", fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

## 1.9 Outlier Detection (IQR Method)

In [ ]:
print("OUTLIER DETECTION (IQR Method)")
print("=" * 65)

outlier_summary = []
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    count = len(outliers)
    pct = count / len(df) * 100
    outlier_summary.append({
        'Feature': col, 'Q1': round(Q1, 4), 'Q3': round(Q3, 4),
        'IQR': round(IQR, 4), 'Lower Bound': round(lower, 4),
        'Upper Bound': round(upper, 4),
        'Outlier Count': count, 'Outlier %': round(pct, 2)
    })

outlier_df = pd.DataFrame(outlier_summary).sort_values('Outlier Count', ascending=False)
print(outlier_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 6))
colors = ['#f87171' if p > 10 else '#fbbf24' if p > 2 else '#34d399'
          for p in outlier_df['Outlier %']]
ax.barh(outlier_df['Feature'], outlier_df['Outlier %'], color=colors, edgecolor='black')
ax.set_xlabel("Outlier Percentage (%)")
ax.set_title("Outlier Percentage per Feature (IQR Method)", fontsize=14, fontweight='bold')
ax.axvline(x=10, color='#f87171', linestyle='--', alpha=0.5, label='>10% High')
ax.axvline(x=2, color='#fbbf24', linestyle='--', alpha=0.5, label='>2% Moderate')
ax.legend()
plt.tight_layout()
plt.show()

## 1.10 Purchase Rate by Categorical Features

In [ ]:
# Purchase rate by Month
month_order = ['Feb', 'Mar', 'May', 'June', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

month_rate = df.groupby('Month')['Revenue'].mean().reindex(month_order)
month_rate.plot(kind='bar', color='#3498db', edgecolor='black', ax=axes[0])
axes[0].set_title('Purchase Rate by Month', fontsize=13)
axes[0].set_ylabel('Purchase Rate')
axes[0].set_xticklabels(month_order, rotation=45)
axes[0].axhline(y=df['Revenue'].mean(), color='red', linestyle='--', label='Overall Rate')
axes[0].legend()

df['Month'].value_counts().reindex(month_order).plot(
    kind='bar', color='#9b59b6', edgecolor='black', ax=axes[1])
axes[1].set_title('Session Count by Month', fontsize=13)
axes[1].set_ylabel('Number of Sessions')
axes[1].set_xticklabels(month_order, rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Purchase rate by VisitorType, Weekend, SpecialDay
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

df.groupby('VisitorType')['Revenue'].mean().sort_values(ascending=False).plot(
    kind='bar', color='#e67e22', edgecolor='black', ax=axes[0])
axes[0].set_title('Purchase Rate by Visitor Type', fontsize=13)
axes[0].set_ylabel('Purchase Rate')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=0)

df.groupby('Weekend')['Revenue'].mean().plot(
    kind='bar', color=['#1abc9c', '#e74c3c'], edgecolor='black', ax=axes[1])
axes[1].set_title('Purchase Rate: Weekday vs Weekend', fontsize=13)
axes[1].set_xticklabels(['Weekday', 'Weekend'], rotation=0)

df.groupby(df['SpecialDay'] > 0)['Revenue'].mean().plot(
    kind='bar', color=['#3498db', '#f39c12'], edgecolor='black', ax=axes[2])
axes[2].set_title('Purchase Rate: Normal vs Special Day', fontsize=13)
axes[2].set_xticklabels(['Normal Day', 'Near Special Day'], rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# Purchase rate by TrafficType, Browser, OS, Region
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for i, col in enumerate(['TrafficType', 'Browser', 'OperatingSystems', 'Region']):
    row, c = i // 2, i % 2
    rate = df.groupby(col)['Revenue'].agg(['mean', 'count']).sort_values('count', ascending=False)
    rate['mean'].plot(kind='bar', color='#2980b9', edgecolor='black', ax=axes[row][c])
    axes[row][c].set_title(f'Purchase Rate by {col}', fontsize=12)
    axes[row][c].set_ylabel('Purchase Rate')
    axes[row][c].axhline(y=df['Revenue'].mean(), color='red', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

## 1.11 Correlation Heatmap — Raw Features

In [ ]:
corr_matrix = df[numeric_cols].corr()

print("Top 10 Absolute Correlations:")
print("-" * 50)
pairs = []
for i in range(len(numeric_cols)):
    for j in range(i + 1, len(numeric_cols)):
        pairs.append((numeric_cols[i], numeric_cols[j], corr_matrix.iloc[i, j]))
pairs.sort(key=lambda x: abs(x[2]), reverse=True)
for c1, c2, v in pairs[:10]:
    print(f"  {c1:<28} {c2:<28} : {v:>7.4f}")

plt.figure(figsize=(14, 12))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5,
            cbar_kws={"shrink": 0.8, "label": "Correlation"})
plt.title("Correlation Heatmap (Numeric Features)", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

# Part 2 — Preprocessing

Cleaning, transforming, and encoding the raw data. Feature creation is deferred
to Part 3 so that all engineered features live in one place.

## 2.1 Remove Duplicates

In [ ]:
data = df.copy()
print(f'Before: {data.shape[0]:,} rows')
data.drop_duplicates(inplace=True)
data.reset_index(drop=True, inplace=True)
print(f'After:  {data.shape[0]:,} rows')
print(f'Removed: {df.shape[0] - data.shape[0]} duplicate rows')

## 2.2 Zero-Inflation Documentation

Several page-count and duration columns contain many legitimate zeros — these
represent sessions where the user did not visit that page category. We keep
them as-is and later create binary activity flags in the feature engineering
step.

In [ ]:
print('Zero-inflation in key columns:')
for col in ['Administrative', 'Informational', 'ProductRelated',
            'Administrative_Duration', 'Informational_Duration']:
    zero_pct = (data[col] == 0).mean() * 100
    print(f'  {col:<30} {zero_pct:.1f}% zeros')

## 2.3 Log Transformation for Skewed Features

In [ ]:
skewed_cols = ['Administrative_Duration', 'Informational_Duration',
               'ProductRelated_Duration', 'PageValues']

fig, axes = plt.subplots(len(skewed_cols), 2, figsize=(14, 4*len(skewed_cols)))
for i, col in enumerate(skewed_cols):
    axes[i][0].hist(data[col], bins=50, color='#e74c3c', alpha=0.7, edgecolor='black')
    axes[i][0].set_title(f'{col} - Original (skew: {data[col].skew():.2f})')
    log_col = np.log1p(data[col])
    axes[i][1].hist(log_col, bins=50, color='#2ecc71', alpha=0.7, edgecolor='black')
    axes[i][1].set_title(f'{col} - Log1p (skew: {log_col.skew():.2f})')

plt.tight_layout()
plt.show()

for col in skewed_cols:
    data[f'{col}_log'] = np.log1p(data[col])

print('Log-transformed columns created:', [f'{c}_log' for c in skewed_cols])

## 2.4 Winsorize Page Counts at 99th Percentile

In [ ]:
for col in ['Administrative', 'Informational', 'ProductRelated']:
    upper = data[col].quantile(0.99)
    n_capped = (data[col] > upper).sum()
    data[col] = data[col].clip(upper=upper)
    print(f'{col}: capped {n_capped} values at {upper:.0f}')

## 2.5 Encoding Categorical Variables

In [ ]:
# Revenue & Weekend: Boolean -> int
data['Revenue'] = data['Revenue'].astype(int)
data['Weekend'] = data['Weekend'].astype(int)
print('Revenue & Weekend: Boolean -> int')

# VisitorType: Group 'Other' into 'New_Visitor', create binary IsReturning
print(f'\nVisitorType before: {data["VisitorType"].value_counts().to_dict()}')
data['VisitorType'] = data['VisitorType'].replace('Other', 'New_Visitor')
data['IsReturning'] = (data['VisitorType'] == 'Returning_Visitor').astype(int)
data.drop(columns='VisitorType', inplace=True)
print('Created IsReturning (1=Returning, 0=New/Other)')

# Month: Ordinal encoding
month_map = {'Feb': 2, 'Mar': 3, 'May': 5, 'June': 6, 'Jul': 7,
             'Aug': 8, 'Sep': 9, 'Oct': 10, 'Nov': 11, 'Dec': 12}
data['Month'] = data['Month'].map(month_map)
print(f'\nMonth: Ordinal encoded {sorted(data["Month"].unique())}')

In [ ]:
def group_rare(series, threshold=0.01):
    counts = series.value_counts(normalize=True)
    rare_cats = counts[counts < threshold].index
    n_rare = series.isin(rare_cats).sum()
    series_clean = series.copy()
    series_clean[series_clean.isin(rare_cats)] = -1
    return series_clean, len(rare_cats), n_rare

for col in ['OperatingSystems', 'Browser', 'TrafficType']:
    n_before = data[col].nunique()
    data[col], n_grouped, n_rows = group_rare(data[col])
    n_after = data[col].nunique()
    print(f'{col}: {n_before} -> {n_after} categories ({n_grouped} rare grouped, {n_rows} rows)')

# Part 3 — Feature Engineering

This section covers the four required tasks: new features, binning, feature
selection, and correlation heatmap on the final feature set.

## 3.1 New Features

We engineer features that capture session intent more directly than the raw
page counts and durations. A buyer and a browser can view the same number of
pages, but the ratios, interactions, and binary activity flags tend to differ
sharply.

In [ ]:
df_fe = data.copy()

# --- Totals across page types ---
df_fe['TotalPages'] = (
    df_fe['Administrative'] + df_fe['Informational'] + df_fe['ProductRelated']
)
df_fe['TotalDuration'] = (
    df_fe['Administrative_Duration']
    + df_fe['Informational_Duration']
    + df_fe['ProductRelated_Duration']
)

# --- Average time per page ---
df_fe['AvgTimePerPage'] = np.where(
    df_fe['TotalPages'] > 0,
    df_fe['TotalDuration'] / df_fe['TotalPages'], 0
)

# --- Product-focus ratios ---
df_fe['ProductPageRatio'] = np.where(
    df_fe['TotalPages'] > 0,
    df_fe['ProductRelated'] / df_fe['TotalPages'], 0
)
df_fe['ProductDurationRatio'] = np.where(
    df_fe['TotalDuration'] > 0,
    df_fe['ProductRelated_Duration'] / df_fe['TotalDuration'], 0
)

# --- Bounce/Exit behavior ---
df_fe['BounceExitDiff'] = df_fe['ExitRates'] - df_fe['BounceRates']
df_fe['BounceExitRatio'] = np.where(
    df_fe['ExitRates'] > 0,
    df_fe['BounceRates'] / df_fe['ExitRates'], 0
)

# --- Activity flags ---
df_fe['HasAdminActivity'] = (df_fe['Administrative'] > 0).astype(int)
df_fe['HasInfoActivity'] = (df_fe['Informational'] > 0).astype(int)
df_fe['HasPageValue'] = (df_fe['PageValues'] > 0).astype(int)

# --- High-intent session flag ---
df_fe['HighIntentSession'] = (
    (df_fe['ProductRelated'] > df_fe['ProductRelated'].median())
    & (df_fe['BounceRates'] < df_fe['BounceRates'].median())
).astype(int)

# --- Session value score: PageValues * (1 - ExitRate) ---
df_fe['SessionValueScore'] = df_fe['PageValues'] * (1 - df_fe['ExitRates'])

# --- SpecialDay binary flag ---
df_fe['IsSpecialDay'] = (df_fe['SpecialDay'] > 0).astype(int)

# --- Season from ordinal month ---
def get_season(month):
    if month in [11, 12]: return 'Holiday'
    elif month in [2, 3]: return 'EarlyYear'
    elif month in [5, 6, 7, 8]: return 'Summer'
    else: return 'Fall'

df_fe['Season'] = df_fe['Month'].apply(get_season)

# --- Weekend x Returning interaction ---
df_fe['WeekendReturning'] = df_fe['Weekend'] * df_fe['IsReturning']

new_feature_cols = [
    'TotalPages', 'TotalDuration', 'AvgTimePerPage',
    'ProductPageRatio', 'ProductDurationRatio',
    'BounceExitDiff', 'BounceExitRatio',
    'HasAdminActivity', 'HasInfoActivity', 'HasPageValue',
    'HighIntentSession', 'SessionValueScore', 'IsSpecialDay',
    'Season', 'WeekendReturning',
]
print(f'Created {len(new_feature_cols)} new features.')
df_fe[new_feature_cols].head()

In [ ]:
# Visualize SessionValueScore distribution by Revenue
fig, ax = plt.subplots(figsize=(10, 5))
for rev, color, label in [(0, COLORS['no'], 'No Purchase'),
                           (1, COLORS['yes'], 'Purchase')]:
    subset = df_fe[df_fe['Revenue'] == rev]['SessionValueScore']
    ax.hist(subset[subset > 0], bins=50, alpha=0.6, color=color,
            label=label, edgecolor='black')
ax.set_title('SessionValueScore Distribution (excluding zeros)', fontsize=13)
ax.set_xlabel('SessionValueScore')
ax.legend()
plt.tight_layout()
plt.show()

## 3.2 Binning Features

Binning turns heavily skewed numerical features into ordered categories that
are more robust to outliers and easier for linear models to exploit.

In [ ]:
# BounceRates -> Low / Medium / High
df_fe['BounceRate_Bin'] = pd.cut(
    df_fe['BounceRates'],
    bins=[-0.001, 0.02, 0.05, 1.0],
    labels=['Low', 'Medium', 'High'],
)

# ExitRates -> Low / Medium / High
df_fe['ExitRate_Bin'] = pd.cut(
    df_fe['ExitRates'],
    bins=[-0.001, 0.03, 0.08, 1.0],
    labels=['Low', 'Medium', 'High'],
)

# PageValues -> Zero / Low / High (most values are zero)
df_fe['PageValue_Bin'] = pd.cut(
    df_fe['PageValues'],
    bins=[-0.001, 0.0001, 20.0, df_fe['PageValues'].max() + 1],
    labels=['Zero', 'Low', 'High'],
)

# TotalDuration -> quartile bins
df_fe['Duration_Bin'] = pd.qcut(
    df_fe['TotalDuration'],
    q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4'],
    duplicates='drop',
)

# ProductRelated -> quartile bins (rank breaks ties)
df_fe['ProductRelated_Bin'] = pd.qcut(
    df_fe['ProductRelated'].rank(method='first'),
    q=4,
    labels=['Q1', 'Q2', 'Q3', 'Q4'],
)

bin_cols = ['BounceRate_Bin', 'ExitRate_Bin', 'PageValue_Bin',
            'Duration_Bin', 'ProductRelated_Bin']

for col in bin_cols:
    print(f'\n{col}:')
    print(df_fe[col].value_counts().sort_index())

In [ ]:
# Sanity check: do the bins separate the target?
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(bin_cols):
    rate = df_fe.groupby(col, observed=True)['Revenue'].mean()
    rate.plot(kind='bar', ax=axes[i], color='steelblue', edgecolor='black')
    axes[i].set_title(f'Purchase rate by {col}')
    axes[i].set_ylabel('P(Revenue=True)')
    axes[i].tick_params(axis='x', rotation=0)

axes[-1].axis('off')
plt.tight_layout()
plt.show()

## 3.3 Feature Selection

Two-stage approach: first drop numerical features that are near-duplicates of
each other (|r| > 0.9), keeping the one more correlated with the target, then
rank the remaining features by both mutual information and Random Forest
importance, keeping features that score well in at least one method.

In [ ]:
# Build the numerical feature matrix for selection.
# Encode remaining categorical features (Season, bins) as integer codes.
df_rank = df_fe.copy()

cat_like = ['Season', 'BounceRate_Bin', 'ExitRate_Bin',
            'PageValue_Bin', 'Duration_Bin', 'ProductRelated_Bin']

for c in cat_like:
    df_rank[c] = df_rank[c].astype('category').cat.codes

X_all = df_rank.drop(columns=['Revenue'])
y_all = df_rank['Revenue']

print('Feature matrix for selection:', X_all.shape)

In [ ]:
# --- Stage 1: correlation filter ---
corr_abs = X_all.corr().abs()
upper = corr_abs.where(np.triu(np.ones(corr_abs.shape), k=1).astype(bool))

target_corr_abs = X_all.apply(lambda col: col.corr(y_all)).abs()

to_drop = set()
for col in upper.columns:
    for row in upper.index:
        if upper.loc[row, col] > 0.9:
            weaker = row if target_corr_abs[row] < target_corr_abs[col] else col
            to_drop.add(weaker)

print(f'Dropping {len(to_drop)} redundant features: {sorted(to_drop)}')
X_reduced = X_all.drop(columns=list(to_drop))
print('After correlation filter:', X_reduced.shape)

In [ ]:
# --- Stage 2: importance ranking ---
mi = mutual_info_classif(X_reduced, y_all, random_state=42)
mi_series = pd.Series(mi, index=X_reduced.columns).sort_values(ascending=False)

rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
rf.fit(X_reduced, y_all)
rf_series = pd.Series(rf.feature_importances_,
                      index=X_reduced.columns).sort_values(ascending=False)

ranking = pd.DataFrame({
    'mutual_info': mi_series,
    'rf_importance': rf_series,
}).sort_values('rf_importance', ascending=False)

ranking

In [ ]:
# Visualize the two rankings side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 8))

mi_series.sort_values().plot(kind='barh', ax=axes[0], color='teal')
axes[0].set_title('Mutual Information with Revenue')
axes[0].set_xlabel('MI score')

rf_series.sort_values().plot(kind='barh', ax=axes[1], color='darkorange')
axes[1].set_title('Random Forest Feature Importance')
axes[1].set_xlabel('Importance')

plt.tight_layout()
plt.show()

In [ ]:
# Keep features that rank in the top 15 of EITHER method
top_mi = set(mi_series.head(15).index)
top_rf = set(rf_series.head(15).index)
selected_features = sorted(top_mi | top_rf)

print(f'Selected {len(selected_features)} features:')
for f in selected_features:
    print(f'  - {f}')

df_selected = df_rank[selected_features + ['Revenue']]
print(f'\nFinal feature matrix shape: {df_selected.shape}')

## 3.4 Correlation Heatmap — Selected Features

Final check on the selected feature set. We want to confirm that no strong
multicollinearity remains and see which features align most strongly with the
target.

In [ ]:
plt.figure(figsize=(12, 10))
corr_sel = df_selected.corr()
mask = np.triu(np.ones_like(corr_sel, dtype=bool), k=1)

sns.heatmap(
    corr_sel, mask=mask, annot=True, fmt='.2f',
    cmap='coolwarm', center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5, cbar_kws={'shrink': 0.8},
)
plt.title('Correlation Heatmap - Selected Features', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation of each selected feature with the target
target_corr_final = (
    df_selected.corr()['Revenue']
    .drop('Revenue')
    .sort_values(key=abs, ascending=False)
)

plt.figure(figsize=(10, 6))
colors = ['steelblue' if v > 0 else 'indianred' for v in target_corr_final]
target_corr_final.plot(kind='barh', color=colors, edgecolor='black')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Correlation of Selected Features with Revenue')
plt.xlabel('Pearson correlation')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

target_corr_final

# Part 4 — Final Preprocessing for Modeling

Train/test split, scaling on continuous features, and SMOTE to address the
class imbalance. This produces the exact datasets handed off to Phase 2.

## 4.1 Train/Test Split

In [ ]:
X = df_selected.drop(columns='Revenue')
y = df_selected['Revenue']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training: {X_train.shape[0]:,} samples ({X_train.shape[0]/len(X)*100:.0f}%)')
print(f'Test:     {X_test.shape[0]:,} samples ({X_test.shape[0]/len(X)*100:.0f}%)')
print(f'\nTrain target: No={int((y_train==0).sum()):,} ({(y_train==0).mean()*100:.1f}%) '
      f'| Yes={int((y_train==1).sum()):,} ({(y_train==1).mean()*100:.1f}%)')
print(f'Test target:  No={int((y_test==0).sum()):,} ({(y_test==0).mean()*100:.1f}%) '
      f'| Yes={int((y_test==1).sum()):,} ({(y_test==1).mean()*100:.1f}%)')

## 4.2 Scaling (StandardScaler)

Fit on training data only to avoid leakage. Binary features are not scaled.

In [ ]:
scaler = StandardScaler()

binary_cols = [col for col in X_train.columns if X_train[col].nunique() <= 2]
continuous_cols = [col for col in X_train.columns if col not in binary_cols]

print(f'Continuous features to scale ({len(continuous_cols)}): {continuous_cols}')
print(f'Binary features not scaled ({len(binary_cols)}): {binary_cols}')

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

if continuous_cols:
    X_train_scaled[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
    X_test_scaled[continuous_cols] = scaler.transform(X_test[continuous_cols])

print('\nScaling complete (fit on train only).')

## 4.3 SMOTE — Address Class Imbalance

In [ ]:
from imblearn.over_sampling import SMOTE

print('BEFORE SMOTE:')
print(f'  No Purchase: {(y_train==0).sum():,}')
print(f'  Purchase:    {(y_train==1).sum():,}')
print(f'  Ratio: {(y_train==0).sum()/(y_train==1).sum():.1f}:1')

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

print(f'\nAFTER SMOTE:')
print(f'  No Purchase: {(y_train_resampled==0).sum():,}')
print(f'  Purchase:    {(y_train_resampled==1).sum():,}')
print(f'  Ratio: {(y_train_resampled==0).sum()/(y_train_resampled==1).sum():.1f}:1')
print(f'  Total: {len(X_train_resampled):,} (was {len(X_train):,})')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

y_train.value_counts().plot(kind='bar', color=PALETTE, edgecolor='black', ax=axes[0])
axes[0].set_title('Before SMOTE', fontsize=13)
axes[0].set_xticklabels(['No Purchase', 'Purchase'], rotation=0)

pd.Series(y_train_resampled).value_counts().plot(
    kind='bar', color=PALETTE, edgecolor='black', ax=axes[1])
axes[1].set_title('After SMOTE', fontsize=13)
axes[1].set_xticklabels(['No Purchase', 'Purchase'], rotation=0)

plt.tight_layout()
plt.show()

## 4.4 Export Datasets for Phase 2

In [ ]:
# Scaled + SMOTE (for Logistic Regression, SVM)
X_train_resampled.to_csv('X_train_scaled_smote.csv', index=False)
pd.Series(y_train_resampled, name='Revenue').to_csv('y_train_smote.csv', index=False)

# Scaled, no SMOTE (for models using class_weight)
X_train_scaled.to_csv('X_train_scaled.csv', index=False)
y_train.to_csv('y_train.csv', index=False)

# Unscaled (for XGBoost / tree-based models)
X_train.to_csv('X_train_unscaled.csv', index=False)

# Test set
X_test_scaled.to_csv('X_test_scaled.csv', index=False)
X_test.to_csv('X_test_unscaled.csv', index=False)
y_test.to_csv('y_test.csv', index=False)

print('All data exported for Phase 2:')
print('\n  For SVM / Logistic Regression:')
print('    -> X_train_scaled_smote.csv + y_train_smote.csv')
print('\n  For XGBoost / Decision Tree:')
print('    -> X_train_unscaled.csv + y_train.csv')
print('\n  Test set (all models):')
print('    -> X_test_scaled.csv / X_test_unscaled.csv + y_test.csv')

## Phase 1 Summary

**EDA:** basic info, missing/duplicates, target balance, univariate and
bivariate distributions, skewness, boxplots, scatter plots, outlier detection,
purchase rates by categorical features, raw correlation heatmap.

**Preprocessing:** removed 125 duplicate rows; documented zero-inflation;
log1p-transformed 4 skewed columns; winsorized page counts at the 99th
percentile; encoded `Revenue`/`Weekend` to int; created `IsReturning` from
`VisitorType`; ordinal-encoded `Month`; grouped rare categories in
`OperatingSystems`, `Browser`, and `TrafficType`.

**Feature Engineering:** created 15 new features capturing session totals,
ratios, bounce/exit behavior, activity flags, intent signals, session value,
seasonality, and interactions. Created 5 binned features from skewed numerical
variables. Applied a two-stage feature selection (correlation filter + MI/RF
ranking). Produced a correlation heatmap of the final selected feature set.

**Final preprocessing:** stratified 80/20 train/test split, StandardScaler on
continuous features (fit on train only), SMOTE applied to training set only,
and export of scaled/unscaled datasets for Phase 2 modeling.